# Order_items Feature Engineering

## Objective

This notebook creates business-oriented features from the cleaned `order_items` dataset.

These engineered features will be used for:

- Exploratory Data Analysis (EDA)
- SQL Analytics
- Power BI Dashboard
- Business Insights & Recommendations

The engineered dataset produced in this notebook will be exported for downstream analysis.

In [6]:
import pandas as pd
import numpy as np

order_items = pd.read_csv(
    "../../03_Python_ETL/Output/order_items_clean.csv",
    parse_dates=["shipping_limit_date"]
)

In [7]:
order_items.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 112650 entries, 0 to 112649
Data columns (total 7 columns):
 #   Column               Non-Null Count   Dtype         
---  ------               --------------   -----         
 0   order_id             112650 non-null  object        
 1   order_item_id        112650 non-null  int64         
 2   product_id           112650 non-null  object        
 3   seller_id            112650 non-null  object        
 4   shipping_limit_date  112650 non-null  datetime64[ns]
 5   price                112650 non-null  float64       
 6   freight_value        112650 non-null  float64       
dtypes: datetime64[ns](1), float64(2), int64(1), object(3)
memory usage: 6.0+ MB


In [13]:
order_items[["price", "freight_value"]].describe()

,price,freight_value
count,112650.000000,112650.000000
mean,120.653739,19.990320
std,183.633928,15.806405
min,0.850000,0.000000
25%,39.900000,13.080000
50%,74.990000,16.260000
75%,134.900000,21.150000
max,6735.000000,409.680000


In [14]:
order_items[["price", "freight_value"]].quantile([0.25, 0.50, 0.75, 0.90, 0.95, 0.99])

,price,freight_value
0.25,39.90,13.080
0.50,74.99,16.260
0.75,134.90,21.150
0.90,229.80,34.041
0.95,349.90,45.120
0.99,890.00,84.520


### Feature 1: freight_to_price_ratio
**Business Objective**

The freight_to_price_ratio measures the shipping cost as a proportion of the product price.

This feature helps identify products where shipping charges are disproportionately high or low relative to the item's value.

In [15]:
#Pyhton Implementation
order_items["freight_to_price_ratio"] = (
    order_items["freight_value"] / order_items["price"]
)

In [25]:
# Validation
order_items[
    ["price", "freight_value", "freight_to_price_ratio"]].head()

,price,freight_value,freight_to_price_ratio
0,58.90,13.29,0.225637
1,239.90,19.93,0.083076
2,199.00,17.87,0.089799
3,12.99,12.79,0.984604
4,199.90,18.14,0.090745


In [17]:
order_items["freight_to_price_ratio"].describe()

count    112650.000000
mean          0.320864
std           0.349894
min           0.000000
25%           0.134034
50%           0.231356
75%           0.393036
max          26.235294
Name: freight_to_price_ratio, dtype: float64

In [18]:
order_items["freight_to_price_ratio"].isnull().sum()

0

In [19]:
np.isinf(order_items["freight_to_price_ratio"]).sum()

0

### Feature 2: order_total_items
**Business Objective**

The order_total_items feature represents the total number of items purchased in each order.

Since each row in the order_items table represents one item, we can count how many rows belong to the same order_id.

In [20]:
order_items["order_total_items"] = (
    order_items.groupby("order_id")["order_item_id"].transform("count"))

In [24]:
#Validation
order_items[
    ["order_id", "order_item_id", "order_total_items"]
].head(10)

,order_id,order_item_id,order_total_items
0,00010242fe8c5a6d1ba2dd792cb16214,1,1
1,00018f77f2f0320c557190d7a144bdd3,1,1
2,000229ec398224ef6ca0657da4fc703e,1,1
3,00024acbcdf0a6daa1e931b038114c75,1,1
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,1
5,00048cc3ae777c65dbb7d2a0634bc1ea,1,1
6,00054e8431b9d7675808bcb819fb4a32,1,1
7,000576fe39319847cbb9d288c5617fa6,1,1
8,0005a1a1728c9d785b8e2b08b904576c,1,1
9,0005f50442cb953dcd1d21e1fb923495,1,1


In [22]:
order_items["order_total_items"].describe()

count    112650.000000
mean          1.395668
std           1.120101
min           1.000000
25%           1.000000
50%           1.000000
75%           1.000000
max          21.000000
Name: order_total_items, dtype: float64

In [23]:
order_items["order_total_items"].isnull().sum()

0

### Feature 3: seller_total_items
**Business Objective**

The seller_total_items feature represents the total number of order items fulfilled by each seller across the entire dataset.

It helps measure the overall sales volume of a seller based on the number of items sold, regardless of the number of orders.

In [27]:
order_items["seller_total_items"] = (
    order_items
    .groupby("seller_id")["order_item_id"]
    .transform("count")
)

In [28]:
# Validation
order_items[
    ["seller_id", "order_item_id", "seller_total_items"]
].head(10)

,seller_id,order_item_id,seller_total_items
0,48436dade18ac8b2bce089ec2a041202,1,151
1,dd7ddc04e1b6c2c614352b383efe2d36,1,143
2,5b51032eddd242adc84c38acab88f23d,1,14
3,9d7a1d34a5052409006425275ba1c2b4,1,16
4,df560393f3a51e74553ab94004ba5c87,1,29
5,6426d21aca402a131fc0a5d0960a3c90,1,23
6,7040e82f899a04d1b434b795a43b4617,1,228
7,5996cddab893a4652a15592fb58ab8db,1,1
8,a416b6a846a11724393025641d4edd5e,1,181
9,ba143b05f0110f0dc71ad71b4466ce92,1,86


In [29]:
order_items["seller_total_items"].describe()

count    112650.000000
mean        426.603444
std         563.122004
min           1.000000
25%          56.000000
50%         173.000000
75%         529.000000
max        2033.000000
Name: seller_total_items, dtype: float64

In [30]:
order_items["seller_total_items"].isnull().sum()

0

In [31]:
# Final Validation
order_items.shape

(112650, 10)

In [32]:
order_items.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 112650 entries, 0 to 112649
Data columns (total 10 columns):
 #   Column                  Non-Null Count   Dtype         
---  ------                  --------------   -----         
 0   order_id                112650 non-null  object        
 1   order_item_id           112650 non-null  int64         
 2   product_id              112650 non-null  object        
 3   seller_id               112650 non-null  object        
 4   shipping_limit_date     112650 non-null  datetime64[ns]
 5   price                   112650 non-null  float64       
 6   freight_value           112650 non-null  float64       
 7   freight_to_price_ratio  112650 non-null  float64       
 8   order_total_items       112650 non-null  int64         
 9   seller_total_items      112650 non-null  int64         
dtypes: datetime64[ns](1), float64(3), int64(3), object(3)
memory usage: 8.6+ MB


In [33]:
order_items.head()

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,freight_to_price_ratio,order_total_items,seller_total_items
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29,0.225637,1,151
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93,0.083076,1,143
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87,0.089799,1,14
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79,0.984604,1,16
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14,0.090745,1,29


In [34]:
order_items.to_csv(
    "../Output/order_items_features.csv",
    index=False
)